# Model Search — Synthetic SVXY −1x

#### **MScFE 690 Capstone -Group 17843**

**Authors**: Edgar NAVA-edgar.nava@grovewm.com & Celestin NYANDWI-celestinnyandwi48@gmail.com

This notebook reproduces the staged parameter search used to select Model 2073. We vary the existing entry and exit rules, allocations and trailing stops, using Development and Validation. The Final Test period is not used in the search or selection.


## 1. Settings and data

Inputs are `VIX_data.csv` in `data/raw/` and `SVXY_synth-ONEx.csv` in `data/processed/`, as defined in the project `config.json`. Run the cells in order.

Each period starts with USD 100,000 in cash. Commission is USD 0.005 per share per side, with 0.10% slippage per side. Fractional shares are allowed; there is no borrowing or interest on cash. Signals and trades use the same Close, as discussed in the Capstone.


In [1]:
from pathlib import Path
from copy import deepcopy
from itertools import product
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project folder containing config.json.")
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
RAW_DATA_DIR = PROJECT_ROOT / CONFIG["raw_data_dir"]
PROCESSED_DATA_DIR = PROJECT_ROOT / CONFIG["processed_data_dir"]
OUTPUT_DIR = PROCESSED_DATA_DIR / "model_search"
SAVE_RESULTS = False

INITIAL_CASH = 100_000.0
COMMISSION_PER_SHARE = 0.005
SLIPPAGE_PCT = 0.001
SESSIONS_PER_YEAR = 252
PERIOD_DATES = {
    "Development": ("2011-10-04", "2020-12-31"),
    "Validation": ("2021-01-04", "2023-12-29"),
}
DATA_CUTOFF = pd.Timestamp("2023-12-29")
MIN_CAGR, MAX_DRAWDOWN = 0.10, 0.45
MIN_SENSITIVITY_PASS = 0.90
RANDOM_SEED, JOINT_VARIATIONS = 20261011, 200

# Starting rules and their historical numeric representation.
STARTING_RULES = {
    "C1": {"enabled": True, "vix_min": 20, "cash_fraction": 0.30,
           "target_nav": 0.30, "trailing_stop_pct": 0.30},
    "C2": {"enabled": True, "vix_min": 25, "cash_fraction": 0.30,
           "target_nav": 0.70, "trailing_stop_pct": 0.25},
    "C3": {"enabled": True, "vix_max": 28, "cash_fraction": 0.50,
           "invested_cash_fraction": 1, "trailing_stop_pct": 0.30},
    "V1": {"enabled": True},
    "V2": {"enabled": True, "vix_min": 18},
    "V3": {"enabled": True, "vix_max": 14},
}

def numeric_rules(rules):
    return {rule: {name: value if name == "enabled" else float(value)
                   for name, value in values.items()} for rule, values in rules.items()}

BASE = {"rules": numeric_rules(STARTING_RULES), "reset_stop_on_buy": True}

def blue_table(frame, caption, formatter=None):
    return (frame.style.format(formatter, na_rep="N/A")
        .set_caption(caption)
        .set_table_styles([
            {"selector": "caption", "props": [("text-align", "left"),
                ("color", "#255c80"), ("font-size", "16px"), ("padding", "8px 0")]},
            {"selector": "th", "props": [("background-color", "#255c80"),
                ("color", "white"), ("text-align", "left"), ("padding", "9px 12px")]},
            {"selector": "td", "props": [("padding", "9px 12px"), ("text-align", "right")]},
            {"selector": "tbody tr:nth-child(even)",
             "props": [("background-color", "#edf3f7")]},
            {"selector": "table", "props": [("border-collapse", "collapse")]},
        ]))

def read_close(filename, directory):
    data = pd.read_csv(directory / filename, usecols=["Date", "Close"], dtype="string")
    data["Date"] = pd.to_datetime(data.Date, format="%Y-%m-%d", errors="raise")
    data = data.loc[data.Date <= DATA_CUTOFF].copy()
    data["Close"] = pd.to_numeric(data.Close, errors="raise").astype(float)
    data = data.sort_values("Date").set_index("Date")
    if data.empty or not data.index.is_unique or not np.isfinite(data.Close).all() or not data.Close.gt(0).all():
        raise ValueError(f"Check dates and closing prices in {filename}.")
    return data

vix = read_close("VIX_data.csv", RAW_DATA_DIR)
svxy = read_close("SVXY_synth-ONEx.csv", PROCESSED_DATA_DIR)
STATE_THRESHOLDS = vix.loc["2004-03-26":DATA_CUTOFF, "Close"].quantile(
    np.array([55, 76, 90, 96]) / 100, interpolation="linear").to_numpy()
vix["State"] = np.searchsorted(STATE_THRESHOLDS, vix.Close.to_numpy(), side="left") + 1
vix["Previous_state"] = vix.State.shift(1)
prices = svxy.join(vix.rename(columns={"Close": "VIX_Close"}), how="left")

PERIODS = {}
period_rows = []
for label, (start, end) in PERIOD_DATES.items():
    frame = prices.loc[start:end]
    if (frame.empty or frame.index[0] != pd.Timestamp(start) or frame.index[-1] != pd.Timestamp(end)
            or frame.isna().any().any() or not frame.index.equals(vix.loc[start:end].index)):
        raise ValueError(f"Check aligned data and period endpoints for {label}.")
    rows = tuple(zip(frame.Close.astype(float).tolist(), frame.VIX_Close.astype(float).tolist(),
                     frame.State.astype(int).tolist(), frame.Previous_state.astype(int).tolist()))
    PERIODS[label] = {"rows": rows, "years": (frame.index[-1] - frame.index[0]).days / 365.25}
    period_rows.append({"Period": label, "Start": start, "End": end, "Sessions": len(frame)})
display(blue_table(pd.DataFrame(period_rows).set_index("Period"), "Search periods"))


,Start,End,Sessions
Period,,,
Development,2011-10-04,2020-12-31,2327
Validation,2021-01-04,2023-12-29,753


## 2. Trading logic and performance

The five VIX states use the 55th, 76th, 90th and 96th percentiles of March 26, 2004–December 29, 2023. Equality belongs to the lower state. The previous state is calculated before dividing the periods.

| Rule | Condition and action |
|:---|:---|
| C1 | S1 → S2, exit flag inactive and VIX above its threshold: buy. |
| C2 | S2 → S3 and VIX above its threshold: buy. |
| C3 | S4 → S3 and VIX below its threshold: buy. |
| V1 | Sell the whole position when Close reaches or falls below the trailing stop. |
| Exit flag | Entering S1 from a higher state while invested activates the flag. |
| V2 | With the flag active, S1 → a higher state and VIX above its threshold: sell. |
| V3 | With the flag active, S1 and VIX below its threshold: sell. |

From cash, entries invest a fraction of available cash, including costs. With a position, C1/C2 buy toward a target share of NAV after costs; C3 invests a fraction of remaining cash. Actual purchases apply the purchasing rule's stop to the whole position and reset the peak when enabled. Purchases do not clear the exit flag. Stops precede other exits; exits precede entries, with no re-entry on the same day.

CAGR uses calendar days / 365.25. Sharpe uses daily net returns, sample standard deviation, 252 sessions and a zero risk-free rate. Maximum drawdown includes initial capital in the running peak. Positions remain valued at the final Close, without a forced sale.


In [2]:
def run_period(data, rules, reset_stop_on_buy):
    c1, c2, c3 = (rules[name] for name in ("C1", "C2", "C3"))
    v1, v2, v3 = (rules[name] for name in ("V1", "V2", "V3"))
    stop_pcts = (np.nan, c1["trailing_stop_pct"], c2["trailing_stop_pct"], c3["trailing_stop_pct"])
    cash, units, flag = float(INITIAL_CASH), 0.0, 0
    peak = active_stop = np.nan
    navs = np.empty(len(data["rows"]))
    tolerance = max(1.0, INITIAL_CASH) * 1e-12
    orders = 0

    for i, (price, vix_close, state, previous) in enumerate(data["rows"]):
        units_before = units
        equity_before = cash + units * price
        sell = False
        if units > 0:
            peak = max(peak, price)
            sell = v1["enabled"] and price <= peak * (1 - active_stop)
        else:
            flag = 0

        if not sell:
            if units > 0 and state == 1 and previous in (2, 3, 4, 5):
                flag = 1
            if units > 0 and flag == 1:
                if v2["enabled"] and previous == 1 and state in (2, 3, 4, 5) and vix_close > v2["vix_min"]:
                    sell = True
                elif v3["enabled"] and state == 1 and vix_close < v3["vix_max"]:
                    sell = True

        if sell:
            execution_price = price * (1 - SLIPPAGE_PCT)
            commission = units * COMMISSION_PER_SHARE
            cash += units * execution_price - commission
            units, flag = 0.0, 0
            peak = active_stop = np.nan
            orders += 1
        else:
            rule = 0
            if c1["enabled"] and previous == 1 and state == 2 and flag == 0 and vix_close > c1["vix_min"]:
                rule, target = 1, c1["target_nav"] if units > 0 else c1["cash_fraction"]
            elif c2["enabled"] and previous == 2 and state == 3 and vix_close > c2["vix_min"]:
                rule, target = 2, c2["target_nav"] if units > 0 else c2["cash_fraction"]
            elif c3["enabled"] and previous == 4 and state == 3 and vix_close < c3["vix_max"]:
                rule, target = 3, c3["invested_cash_fraction"] if units > 0 else c3["cash_fraction"]
            if rule:
                execution_price = price * (1 + SLIPPAGE_PCT)
                all_in_price = execution_price + COMMISSION_PER_SHARE
                if units == 0 or rule == 3 or target == 1.0:
                    quantity = target * cash / all_in_price
                else:
                    gap = max(0.0, target * equity_before - units * price)
                    quantity = min(cash / all_in_price,
                        gap / (price + target * (price * SLIPPAGE_PCT + COMMISSION_PER_SHARE)))
                if quantity * all_in_price > tolerance:
                    commission = quantity * COMMISSION_PER_SHARE
                    cash -= quantity * execution_price + commission
                    units += quantity
                    active_stop = stop_pcts[rule]
                    if units_before == 0 or reset_stop_on_buy:
                        peak = price
                    orders += 1
        if -tolerance <= cash < 0:
            cash = 0.0
        if cash < 0 or units < 0:
            raise ArithmeticError("Negative cash or units.")
        if units == 0:
            flag = 0
        navs[i] = cash + units * price

    returns = np.empty(len(navs))
    returns[0] = navs[0] / INITIAL_CASH - 1
    returns[1:] = navs[1:] / navs[:-1] - 1
    drawdown = navs / np.maximum(np.maximum.accumulate(navs), INITIAL_CASH) - 1
    volatility = float(returns.std(ddof=1))
    final = float(navs[-1])
    return {
        "NAV": final, "CAGR": (final / INITIAL_CASH) ** (1 / data["years"]) - 1,
        "Sharpe": float(np.sqrt(SESSIONS_PER_YEAR) * returns.mean() / volatility) if volatility > 0 else 0.0,
        "SharpeDefined": volatility > 0, "MaxDD": float(drawdown.min()), "Orders": orders,
    }

def evaluate_config(config, periods):
    rules = numeric_rules(config["rules"])
    return {label: run_period(data, rules, config["reset_stop_on_buy"])
            for label, data in periods.items()}

def passes_filters(metrics):
    return all(metrics[p]["CAGR"] > MIN_CAGR and metrics[p]["MaxDD"] > -MAX_DRAWDOWN for p in PERIODS)


## 3. Staged parameter search

We first vary rule activation and individual parameters, then exits and allocations, entry thresholds, stops and final sizing. Later stages use promising configurations from earlier stages, ranked by Development Sharpe while retaining different parameter combinations. The grids and grouping criteria are shown below.

Candidates must have CAGR > 10% and maximum drawdown smaller than 45% in both Development and Validation. These are search filters. Trial numbering follows the original run; recorded trials are not all economically distinct strategies.


In [3]:
records, known = [], {}

def signature(config):
    rules = deepcopy(config['rules'])
    for rule, params in list(rules.items()):
        if not params['enabled']:
            rules[rule] = {'enabled': False}
        elif not rules['V1']['enabled']:
            params.pop('trailing_stop_pct', None)
    effective = {'rules': rules}
    if rules['V1']['enabled']:
        effective['reset_stop_on_buy'] = config['reset_stop_on_buy']
    return json.dumps(effective, sort_keys=True)

def evaluate(config, stage, label):
    key = hashlib.sha256(signature(config).encode()).hexdigest()[:16]
    if key in known:
        prior = known[key]
        if stage not in prior['stages']:
            prior['stages'].append(stage)
        return prior
    results = evaluate_config(config, PERIODS)
    row = {'key': key, 'trial': len(records) + 1, 'stages': [stage], 'label': label, 'config': deepcopy(config), 'metrics': results}
    row['feasible'] = passes_filters(results)
    records.append(row)
    known[key] = row
    return row


In [4]:
probes=[]
def add(name,changes={},reset=True):
    r=deepcopy(STARTING_RULES)
    for k,v in changes.items():
        rule,param=k.split('.')
        r[rule][param]=v
    probes.append({'name':name,'rules':r,'reset':reset})
add('baseline')
for rid in STARTING_RULES: add('disable_'+rid,{rid+'.enabled':False})
add('disable_V2_V3',{'V2.enabled':False,'V3.enabled':False})
add('no_reset',reset=False)
for x in [10.,11.,12.,13.,15.,16.]: add('V3_'+str(x),{'V3.vix_max':x})
for x in [19.,20.,22.,25.,28.]: add('V2_'+str(x),{'V2.vix_min':x})
for x in [18.,19.,21.,22.]: add('C1_threshold_'+str(x),{'C1.vix_min':x})
for x in [22.5,23.,24.,26.,27.,28.]: add('C2_threshold_'+str(x),{'C2.vix_min':x})
for x in [24.,25.,26.,27.,29.]: add('C3_threshold_'+str(x),{'C3.vix_max':x})
for x in [.1,.2,.4,.5,.7]: add('C1_allocation_'+str(x),{'C1.cash_fraction':x,'C1.target_nav':x})
for x in [.1,.2,.4,.5,.7]: add('C2_entry_'+str(x),{'C2.cash_fraction':x})
for x in [.3,.5,.9,1.]: add('C2_target_'+str(x),{'C2.target_nav':x})
for x in [.3,.4,.6,.7,.8,1.]: add('C3_entry_'+str(x),{'C3.cash_fraction':x})
for x in [.3,.5,.7]: add('C3_add_'+str(x),{'C3.invested_cash_fraction':x})


In [5]:
# Rule activation and reference configurations.
stage = 'ablation'
evaluate(BASE, stage, 'Previous winner 30/25/30')
for entry in product([True, False], repeat=3):
    if not any(entry):
        continue
    for exits in product([True, False], repeat=3):
        for reset in [True, False]:
            config = deepcopy(BASE)
            for name, on in zip(('C1', 'C2', 'C3', 'V1', 'V2', 'V3'), entry + exits):
                config['rules'][name]['enabled'] = on
            config['reset_stop_on_buy'] = reset
            label = 'ON ' + ','.join((k for k, r in config['rules'].items() if r['enabled'])) + f'; reset={reset}'
            evaluate(config, stage, label)
for name, c2 in [('Rules 20/20/20', 0.3), ('Original S1 20/20/20', 0.7)]:
    config = deepcopy(BASE)
    for r in ('C1', 'C2', 'C3'):
        config['rules'][r]['trailing_stop_pct'] = 0.2
    config['rules']['C2']['cash_fraction'] = c2
    evaluate(config, stage, name)
for probe in probes:
    config = deepcopy(BASE)
    config['rules'] = probe['rules']
    config['reset_stop_on_buy'] = probe['reset']
    row = evaluate(config, 'single_parameter', probe['name'])


In [6]:
# Exit thresholds and C3 allocations.
stage = 'exits_allocation'
for v2, v3, entry, add, stop_mode in product([18.0, 18.5, 19.0, 19.5, 20.0], [12.0, 14.0, 15.0, 16.0, 17.0], [0.2, 0.3, 0.5], [0.3, 0.5, 0.7, 1.0], ['reset', 'keep_peak', 'off']):
    config = deepcopy(BASE)
    config['rules']['V2']['vix_min'] = v2
    config['rules']['V3']['vix_max'] = v3
    config['rules']['C3']['cash_fraction'] = entry
    config['rules']['C3']['invested_cash_fraction'] = add
    config['rules']['V1']['enabled'] = stop_mode != 'off'
    config['reset_stop_on_buy'] = stop_mode != 'keep_peak'
    evaluate(config, stage, f'V2>{v2:g} V3<{v3:g} C3cash={entry:g}/{add:g} stop={stop_mode}')


In [7]:
# Entry thresholds and independent stops.
stage = 'entry_and_stops'
ranked = sorted([r for r in records if r['feasible']], key=lambda r: r['metrics']['Development']['Sharpe'], reverse=True)
seeds = []
for stop_on in (True, False):
    groups = set()
    for r in ranked:
        rules = r['config']['rules']
        group = (rules['C3']['cash_fraction'], rules['C3']['invested_cash_fraction'], rules['V3']['vix_max'])
        if rules['V1']['enabled'] == stop_on and group not in groups:
            seeds.append(r)
            groups.add(group)
            if len(groups) == 4:
                break
for seed in seeds:
    for c1, c2, c3, mask in product([18.0, 19.0, 20.0], [23.0, 24.0, 25.0, 27.0], [26.0, 28.0, 29.0], ['all', 'C1off', 'C2off', 'C1C2off']):
        config = deepcopy(seed['config'])
        config['rules']['C1']['vix_min'] = c1
        config['rules']['C2']['vix_min'] = c2
        config['rules']['C3']['vix_max'] = c3
        config['rules']['C1']['enabled'] = mask not in ('C1off', 'C1C2off')
        config['rules']['C2']['enabled'] = mask not in ('C2off', 'C1C2off')
        evaluate(config, stage, f"Entry thresholds {c1:g}/{c2:g}/{c3:g} {mask}; seed {seed['trial']}")
ranked = sorted([r for r in records if r['feasible'] and r['config']['rules']['V1']['enabled']], key=lambda r: r['metrics']['Development']['Sharpe'], reverse=True)
groups = set()
seeds = []
for r in ranked:
    rule = r['config']['rules']
    group = (rule['C1']['vix_min'], rule['C2']['vix_min'], rule['C3']['vix_max'], rule['C3']['cash_fraction'], rule['C3']['invested_cash_fraction'])
    if group not in groups:
        groups.add(group)
        seeds.append(r)
        if len(seeds) == 4:
            break
for seed in seeds:
    for stops in product([0.15, 0.2, 0.25, 0.3, 0.35, 0.4], repeat=3):
        config = deepcopy(seed['config'])
        for name, value in zip(('C1', 'C2', 'C3'), stops):
            config['rules'][name]['trailing_stop_pct'] = value
        evaluate(config, 'stop_tuning', f"Stops {stops}; seed {seed['trial']}")


In [8]:
# Position sizing and final C3 stop adjustment.
stage = 'sizing'
ranked = sorted([r for r in records if r['feasible']], key=lambda r: r['metrics']['Development']['Sharpe'], reverse=True)
groups = set()
seeds = []
for r in ranked:
    rr = r['config']['rules']
    group = (rr['C1']['enabled'], rr['C2']['trailing_stop_pct'], rr['C3']['trailing_stop_pct'], rr['V1']['enabled'])
    if group not in groups:
        groups.add(group)
        seeds.append(r)
        if len(seeds) == 4:
            break
for seed in seeds:
    for c2flat, c2target, c3flat, c3add in product([0.15, 0.3, 0.5], [0.3, 0.5, 0.7], [0.1, 0.2, 0.3], [0.3, 0.5, 0.7, 0.9]):
        config = deepcopy(seed['config'])
        config['rules']['C2']['cash_fraction'] = c2flat
        config['rules']['C2']['target_nav'] = c2target
        config['rules']['C3']['cash_fraction'] = c3flat
        config['rules']['C3']['invested_cash_fraction'] = c3add
        evaluate(config, stage, f"Sizing C2={c2flat:g}/{c2target:g} C3={c3flat:g}/{c3add:g}; seed {seed['trial']}")
top = sorted([r for r in records if r['feasible'] and r['config']['rules']['V1']['enabled']], key=lambda r: r['metrics']['Development']['Sharpe'], reverse=True)[:6]
for seed in top:
    for c3stop in [0.3, 0.35, 0.4, 0.45]:
        config = deepcopy(seed['config'])
        config['rules']['C3']['trailing_stop_pct'] = c3stop
        evaluate(config, stage, f"C3 stop={c3stop:g}; seed {seed['trial']}")


In [9]:
stage_names = {
    "ablation": "Rule activation and references",
    "single_parameter": "Individual parameter probes",
    "exits_allocation": "Exit thresholds and allocations",
    "entry_and_stops": "Entry thresholds",
    "stop_tuning": "Trailing stops",
    "sizing": "Final sizing and C3 stop adjustment",
}
stage_summary = pd.DataFrame([
    {"Stage": label, "Trials": sum(r["stages"][0] == stage for r in records),
     "Passing both periods": sum(r["stages"][0] == stage and r["feasible"] for r in records)}
    for stage, label in stage_names.items()
]).set_index("Stage")
display(blue_table(stage_summary, "Search summary — first stage in which each trial was recorded"))


,Trials,Passing both periods
Stage,,
Rule activation and references,86,0
Individual parameter probes,58,0
Exit thresholds and allocations,897,232
Entry thresholds,472,258
Trailing stops,320,177
Final sizing and C3 stop adjustment,446,406


## 4. Parameter sensitivity and selection

We shortlist eight distinct performance results ranked by the lower of Development and Validation Sharpe, adding the highest Development Sharpe and CAGR candidates if not already represented. Development CAGR breaks ties in the balanced ranking.

For each candidate, we vary active parameters individually and in 200 randomly sampled joint combinations: ±0.5 VIX points, ±5 percentage points of allocation and ±2.5 percentage points of trailing stop. The random seed for the joint variations is 20261011. At least 90% of these variations must pass both-period filters. Among qualifying candidates, we select the highest lower-period Sharpe, using Development CAGR to break ties. The neighboring variations assess sensitivity; they are not additional winner candidates.


In [10]:
feasible = [r for r in records if r['feasible']]
balanced = lambda r: min((r['metrics'][p]['Sharpe'] for p in PERIODS))
ranked = sorted(feasible, key=lambda r: (balanced(r), r['metrics']['Development']['CAGR']), reverse=True)
shortlist = []
seen = set()
def add(r):
    fingerprint = tuple((round(r['metrics'][p][m], 10) for p in PERIODS for m in ('CAGR', 'Sharpe', 'MaxDD')))
    if fingerprint not in seen:
        shortlist.append(r)
        seen.add(fingerprint)
for r in ranked:
    add(r)
    if len(shortlist) == 8:
        break
add(max(feasible, key=lambda r: r['metrics']['Development']['Sharpe']))
add(max(feasible, key=lambda r: r['metrics']['Development']['CAGR']))
def parameters(config):
    for rule, values in config['rules'].items():
        if not values['enabled']:
            continue
        for parameter, value in values.items():
            if parameter == 'enabled':
                continue
            if parameter == 'trailing_stop_pct' and (not config['rules']['V1']['enabled']):
                continue
            delta = 0.5 if parameter.startswith('vix_') else 0.025 if parameter == 'trailing_stop_pct' else 0.05
            yield (rule, parameter, value, delta)
rows = []
summaries = []
for candidate in shortlist:
    config = candidate['config']
    params = list(parameters(config))
    experiments = []
    for rule, param, value, delta in params:
        for direction in [-1, 1]:
            c = deepcopy(config)
            c['rules'][rule][param] = float(np.clip(value + direction * delta, 0.0001, 0.9999) if not param.startswith('vix_') else value + direction * delta)
            experiments.append(('single', f'{rule}.{param}{direction:+}', c))
    rng = np.random.default_rng(RANDOM_SEED)
    for i in range(JOINT_VARIATIONS):
        c = deepcopy(config)
        for rule, param, value, delta in params:
            v = value + rng.uniform(-delta, delta)
            c['rules'][rule][param] = float(np.clip(v, 0.0001, 0.9999) if not param.startswith('vix_') else v)
        experiments.append(('joint', str(i + 1), c))
    outcomes = []
    for kind, label, c in experiments:
        results = evaluate_config(c, PERIODS)
        passes = passes_filters(results)
        row = {'key': candidate['key'], 'kind': kind, 'case': label, 'feasible': passes}
        row.update({f'{p}_{m}': results[p][m] for p in PERIODS for m in ('CAGR', 'Sharpe', 'MaxDD')})
        rows.append(row)
        outcomes.append(row)
    frame = pd.DataFrame(outcomes)
    summary = {'key': candidate['key'], 'label': f"Trial {candidate['trial']}", 'total': len(frame), 'feasible': int(frame.feasible.sum()), 'fraction_feasible': float(frame.feasible.mean()), 'single_total': int(frame.kind.eq('single').sum()), 'single_feasible': int(frame.loc[frame.kind.eq('single'), 'feasible'].sum()), 'joint_total': JOINT_VARIATIONS, 'joint_feasible': int(frame.loc[frame.kind.eq('joint'), 'feasible'].sum())}
    for short, p in [('dev', 'Development'), ('val', 'Validation')]:
        summary[f'min_{short}_cagr'] = float(frame[f'{p}_CAGR'].min())
        summary[f'max_{short}_cagr'] = float(frame[f'{p}_CAGR'].max())
        summary[f'worst_{short}_dd'] = float(frame[f'{p}_MaxDD'].min())
        summary[f'median_{short}_cagr'] = float(frame[f'{p}_CAGR'].median())
    summaries.append(summary)
robust = {s['key']: s for s in summaries}
eligible = [r for r in shortlist if robust[r['key']]['fraction_feasible'] >= MIN_SENSITIVITY_PASS]
assert eligible, 'No shortlisted candidate meets the declared 90% neighborhood criterion.'
selected = max(eligible, key=lambda r: (balanced(r), r['metrics']['Development']['CAGR']))


## 5. Selected model

The tables summarize the shortlisted candidates and the selected configuration. Calmar is reported as CAGR divided by the absolute maximum drawdown over the same period; it was not a selection criterion. Detailed performance and comparison with buy and hold are presented in Notebook 08.


In [11]:
sensitivity_table = pd.DataFrame([
    {"Trial": r["trial"], "Development Sharpe": r["metrics"]["Development"]["Sharpe"],
     "Validation Sharpe": r["metrics"]["Validation"]["Sharpe"],
     "Passing variations": f'{robust[r["key"]]["feasible"]}/{robust[r["key"]]["total"]}',
     "Pass rate": robust[r["key"]]["fraction_feasible"],
     "Selected": "Yes" if r["key"] == selected["key"] else ""}
    for r in shortlist
]).set_index("Trial")
display(blue_table(sensitivity_table, "Shortlist and parameter sensitivity", {
    "Development Sharpe": "{:.3f}", "Validation Sharpe": "{:.3f}", "Pass rate": "{:.2%}"}))
display(Markdown(f'**Selected model: {selected["trial"]}.** '
    f'{len(records):,} recorded search trials; {sum(r["feasible"] for r in records):,} pass both-period filters.'))

rules = selected["config"]["rules"]
rule_rows = []
for name, values in rules.items():
    if not values["enabled"]:
        description = "Disabled"
    elif name in ("C1", "C2", "C3"):
        threshold = "vix_max" if name == "C3" else "vix_min"
        description = f'VIX {"<" if name == "C3" else ">"} {values[threshold]:g}; '
        description += f'from cash: {values["cash_fraction"]:.0%}; '
        description += (f'with a position: {values["invested_cash_fraction"]:.0%} of remaining cash; '
                        if name == "C3" else f'with a position: target {values["target_nav"]:.0%} of NAV; ')
        description += f'trailing stop: {values["trailing_stop_pct"]:.0%}'
    elif name == "V1":
        description = "Trailing stop enabled"
    else:
        threshold = "vix_min" if name == "V2" else "vix_max"
        description = f'Exit flag active; VIX {">" if name == "V2" else "<"} {values[threshold]:g}'
    rule_rows.append({"Rule": name, "Selected parameters": description})
rule_rows.append({"Rule": "Peak reset on purchase", "Selected parameters": str(selected["config"]["reset_stop_on_buy"])})
display(blue_table(pd.DataFrame(rule_rows).set_index("Rule"), "Selected rule settings"))

metric_rows = []
for period, values in selected["metrics"].items():
    metric_rows.append({"Period": period, "CAGR": values["CAGR"],
        "Sharpe": values["Sharpe"] if values["SharpeDefined"] else np.nan,
        "Calmar": values["CAGR"] / abs(values["MaxDD"]) if values["MaxDD"] < 0 else np.nan,
        "Maximum drawdown": abs(values["MaxDD"]), "Final equity (USD)": values["NAV"]})
selected_metrics = pd.DataFrame(metric_rows).set_index("Period")
display(blue_table(selected_metrics, "Selected model — Development and Validation", {
    "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "Calmar": "{:.3f}",
    "Maximum drawdown": "{:.2%}", "Final equity (USD)": "{:,.2f}"}))


,Development Sharpe,Validation Sharpe,Passing variations,Pass rate,Selected
Trial,,,,,
2073,0.912,0.942,198/220,90.00%,Yes
2180,0.898,0.942,210/228,92.11%,
2060,0.892,0.956,140/220,63.64%,
2109,0.881,0.977,198/220,90.00%,
2077,0.878,0.952,199/220,90.45%,
2074,0.875,0.981,194/220,88.18%,
2167,0.872,0.956,154/228,67.54%,
2216,0.869,0.977,211/228,92.54%,
1859,0.917,0.694,191/220,86.82%,


**Selected model: 2073.** 2,279 recorded search trials; 1,073 pass both-period filters.

,Selected parameters
Rule,
C1,Disabled
C2,VIX > 23; from cash: 15%; with a position: target 70% of NAV; trailing stop: 20%
C3,VIX < 28; from cash: 10%; with a position: 50% of remaining cash; trailing stop: 40%
V1,Trailing stop enabled
V2,Exit flag active; VIX > 19
V3,Exit flag active; VIX < 15
Peak reset on purchase,True


,CAGR,Sharpe,Calmar,Maximum drawdown,Final equity (USD)
Period,,,,,
Development,18.94%,0.912,0.831,22.79%,"496,956.75"
Validation,31.92%,0.942,1.014,31.49%,"228,423.69"


### Brief interpretation

Six candidates met the 90% sensitivity requirement. Among them, Model 2073 had the highest minimum Sharpe ratio across Development and Validation (0.912), so we selected it. The model met the CAGR and drawdown requirements in both periods. We retained its original trial number.


## 6. Optional output files

Set `SAVE_RESULTS = True` to save the search summary, sensitivity summary, selected parameters and selected metrics under `data/processed/model_search/`, using the configured processed-data folder.


In [12]:
if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stage_summary.to_csv(OUTPUT_DIR / "search_summary.csv")
    sensitivity_table.to_csv(OUTPUT_DIR / "sensitivity_summary.csv")
    selected_metrics.to_csv(OUTPUT_DIR / "selected_metrics.csv")
    result = {
        "trial": selected["trial"], "key": selected["key"], "config": selected["config"],
        "periods": PERIOD_DATES, "state_thresholds": STATE_THRESHOLDS.tolist(),
        "initial_cash": INITIAL_CASH, "commission_per_share": COMMISSION_PER_SHARE,
        "slippage_pct": SLIPPAGE_PCT, "metrics": selected["metrics"],
        "sensitivity": robust[selected["key"]],
    }
    (OUTPUT_DIR / "selected_model.json").write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")
    print("Saved four result files in the configured model_search folder.")
